In [1]:
import pandas as pd

data = pd.read_csv("trainingData.csv")

print("行数和列数：", data.shape)
data[["WAP001", "WAP002", "WAP003", "BUILDINGID", "FLOOR"]].head()

行数和列数： (19937, 529)


,WAP001,WAP002,WAP003,BUILDINGID,FLOOR
0,100,100,100,1,2
1,100,100,100,1,2
2,100,100,100,1,2
3,100,100,100,1,2
4,100,100,100,0,0


In [2]:
# 找到所有表示 Wi-Fi 热点的列
wifi_columns = [col for col in data.columns if col.startswith("WAP")]

# 取第一条扫描，只保留实际检测到的信号
first_scan = data.loc[0, wifi_columns]
detected = first_scan[first_scan != 100]

print("数据集中的热点总数：", len(wifi_columns))
print("第一条扫描实际检测到的热点数：", len(detected))
print("这条记录的正确答案：")
print(data.loc[0, ["BUILDINGID", "FLOOR"]])

detected.head(10)

数据集中的热点总数： 520
第一条扫描实际检测到的热点数： 17
这条记录的正确答案：
BUILDINGID    1.0
FLOOR         2.0
Name: 0, dtype: float64


WAP090   -67.0
WAP091   -68.0
WAP103   -68.0
WAP104   -68.0
WAP105   -78.0
WAP106   -79.0
WAP125   -92.0
WAP126   -95.0
WAP166   -91.0
WAP167   -89.0
Name: 0, dtype: float64

In [3]:
# X 是模型的输入：只取 Wi-Fi 信号列
X = data[wifi_columns].copy()

# y 是正确答案：把楼栋和楼层合成一个标签
y = data["BUILDINGID"].astype(str) + "_" + data["FLOOR"].astype(str)

print("输入表的形状：", X.shape)
print("答案的数量：", len(y))
print("不同的楼栋＋楼层组合数：", y.nunique())
print("前 5 条记录的答案：")
print(y.head())

输入表的形状： (19937, 520)
答案的数量： 19937
不同的楼栋＋楼层组合数： 13
前 5 条记录的答案：
0    1_2
1    1_2
2    1_2
3    1_2
4    0_0
dtype: object


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

# 同一栋楼、同一层、同一个坐标的记录归到同一组
location_group = data.groupby(
    ["BUILDINGID", "FLOOR", "LONGITUDE", "LATITUDE"]
).ngroup()

# 计划分成五份，取其中一份验证，其余四份训练
splitter = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=42
)
train_idx, val_idx = next(splitter.split(X, y, groups=location_group))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

shared_locations = set(location_group.iloc[train_idx]) & set(location_group.iloc[val_idx])

print("训练记录数：", len(X_train))
print("验证记录数：", len(X_val))
print("训练和验证的类别数：", y_train.nunique(), y_val.nunique())
print("两组共享的采集位置数：", len(shared_locations))

训练记录数： 15950
验证记录数： 3987
训练和验证的类别数： 13 13
两组共享的采集位置数： 0


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 用一个低值编码“未检测到”，原始数据里的 100 保持原样
X_train_model = X_train.replace(100, -110)
X_val_model = X_val.replace(100, -110)

# 标准化参数只从训练集计算，再用同一套参数处理验证集
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_model, y_train)

pred_original = model.predict(X_val_model)
acc_original = accuracy_score(y_val, pred_original)

print(f"原始验证记录的定位准确率：{acc_original:.2%}")
print(f"猜对了 {(pred_original == y_val).sum()} / {len(y_val)} 条")

原始验证记录的定位准确率：96.97%
猜对了 3866 / 3987 条


In [6]:
import numpy as np

# 从训练集统计每个热点被检测到多少次
detection_counts = X_train.ne(100).sum()
available_hotspots = detection_counts[detection_counts > 0].index
top5_hotspots = detection_counts.nlargest(5).index.tolist()

def evaluate_outage(hotspots):
    # 先数有多少条扫描原本能看到至少一个将被停用的热点
    affected = X_val[hotspots].ne(100).any(axis=1).sum()

    # 在验证集的副本上模拟故障，再沿用之前的输入编码
    X_outage = X_val.copy()
    X_outage.loc[:, hotspots] = 100
    predictions = model.predict(X_outage.replace(100, -110))

    return accuracy_score(y_val, predictions), affected

# 随机抽 20 组，每组停用 5 个热点，最后取平均
rng = np.random.default_rng(42)
random_results = []

for _ in range(20):
    hotspots = rng.choice(available_hotspots, size=5, replace=False).tolist()
    random_results.append(evaluate_outage(hotspots))

acc_random, affected_random = np.mean(random_results, axis=0)
acc_top5, affected_top5 = evaluate_outage(top5_hotspots)

print("最常被检测到的 5 个热点：", top5_hotspots)

pd.DataFrame({
    "验证条件": ["原始记录", "随机停用 5 个（20 组平均）", "停用 5 个常见热点"],
    "定位准确率": [f"{a:.2%}" for a in [acc_original, acc_random, acc_top5]],
    "比原始下降（百分点）": [
        f"{100 * (acc_original - a):.2f}"
        for a in [acc_original, acc_random, acc_top5]
    ],
    "实际少了信号的记录数": [0, round(affected_random, 1), affected_top5],
})

最常被检测到的 5 个热点： ['WAP087', 'WAP496', 'WAP502', 'WAP517', 'WAP078']


,验证条件,定位准确率,比原始下降（百分点）,实际少了信号的记录数
0,原始记录,96.97%,0.00,0.0
1,随机停用 5 个（20 组平均）,96.73%,0.23,695.0
2,停用 5 个常见热点,96.51%,0.45,1576.0


In [7]:
# 找出原本能看到至少一个常见热点的扫描
affected = X_val[top5_hotspots].ne(100).any(axis=1)

# 模拟常见热点故障，用之前训练好的模型预测
X_top5 = X_val.copy()
X_top5.loc[:, top5_hotspots] = 100
pred_top5 = model.predict(X_top5.replace(100, -110))

was_correct = pred_original == y_val.to_numpy()
now_correct = pred_top5 == y_val.to_numpy()

print("受影响的扫描数：", affected.sum())
print("从猜对变成猜错：", (affected & was_correct & ~now_correct).sum())
print("从猜错变成猜对：", (affected & ~was_correct & now_correct).sum())

受影响的扫描数： 1576
从猜对变成猜错： 25
从猜错变成猜对： 7


In [8]:
# 找到第一条原来猜对、故障后猜错的记录
new_errors = affected.to_numpy() & was_correct & ~now_correct
i = np.flatnonzero(new_errors)[0]

# 查看这条记录里，哪些原本可见的热点被停用了
signals = X_val.iloc[i][top5_hotspots]
lost_signals = signals[signals != 100]

print("真实位置：", y_val.iloc[i])
print("故障前预测：", pred_original[i])
print("故障后预测：", pred_top5[i])
print("实际丢失的热点数：", len(lost_signals))

lost_signals.rename("故障前的信号强度（dBm）").to_frame()

真实位置： 2_3
故障前预测： 2_3
故障后预测： 2_4
实际丢失的热点数： 4


,故障前的信号强度（dBm）
WAP087,-85
WAP496,-77
WAP502,-76
WAP517,-87


In [9]:
# 找出每列对应的热点，并标记哪几列是常见热点
values = X_val.to_numpy()
is_top5 = X_val.columns.isin(top5_hotspots)
visible = values != 100

# 每条扫描在常见热点故障下实际丢失的信号数
lost_count = visible[:, is_top5].sum(axis=1)

# 只比较确实丢了信号，而且有足够其他信号可删的扫描
other_count = visible[:, ~is_top5].sum(axis=1)
rows = np.flatnonzero((lost_count > 0) & (other_count >= lost_count))

X_compare = X_val.iloc[rows]
y_compare = y_val.iloc[rows]

# 在同一批扫描上，重复 20 轮随机删点
rng = np.random.default_rng(42)
random_scores = []

for _ in range(20):
    masked = X_compare.to_numpy(copy=True)
    for j, row in enumerate(rows):
        candidates = np.flatnonzero(visible[row] & ~is_top5)
        removed = rng.choice(candidates, size=lost_count[row], replace=False)
        masked[j, removed] = 100

    X_random_matched = pd.DataFrame(masked, columns=X_compare.columns)
    pred = model.predict(X_random_matched.replace(100, -110))
    random_scores.append(accuracy_score(y_compare, pred))

print("可比较的扫描数：", len(rows))
print(f"原始记录：{accuracy_score(y_compare, pred_original[rows]):.2%}")
print(f"随机删同样多的其他信号（20 轮平均）：{np.mean(random_scores):.2%}")
print(f"删除常见热点信号：{accuracy_score(y_compare, pred_top5[rows]):.2%}")

可比较的扫描数： 1576
原始记录：98.03%
随机删同样多的其他信号（20 轮平均）：93.91%
删除常见热点信号：96.89%


In [10]:
results = []

def add_result(condition, k, accuracy, affected_count):
    results.append({
        "故障条件": condition,
        "停用热点数": k,
        "准确率（%）": 100 * accuracy,
        "比原始下降（百分点）": 100 * (acc_original - accuracy),
        "实际丢失信号的扫描数": affected_count,
    })

add_result("原始记录", 0, acc_original, 0)

for k in [5, 10, 20]:
    # 随机停用：每档抽 20 组，每组停用 k 个热点
    rng = np.random.default_rng(42)
    random_runs = []

    for _ in range(20):
        hotspots = rng.choice(available_hotspots, size=k, replace=False).tolist()
        random_runs.append(evaluate_outage(hotspots))

    avg_accuracy, avg_affected = np.mean(random_runs, axis=0)
    add_result("随机停用（20 组平均）", k, avg_accuracy, avg_affected)

    # 常见热点停用：按训练集里的检测次数挑前 k 个
    common_hotspots = detection_counts.nlargest(k).index.tolist()
    accuracy, affected_count = evaluate_outage(common_hotspots)
    add_result("停用常见热点", k, accuracy, affected_count)

outage_table = pd.DataFrame(results)
outage_table.round(2)

,故障条件,停用热点数,准确率（%）,比原始下降（百分点）,实际丢失信号的扫描数
0,原始记录,0,96.97,0.00,0.00
1,随机停用（20 组平均）,5,96.73,0.23,695.05
2,停用常见热点,5,96.51,0.45,1576.00
3,随机停用（20 组平均）,10,96.31,0.66,1426.75
4,停用常见热点,10,94.48,2.48,2144.00
5,随机停用（20 组平均）,20,95.39,1.57,2232.70
6,停用常见热点,20,90.12,6.85,2371.00


In [11]:
def mean_lost_signals(hotspots):
    # 对每条扫描，数一数原本能看到几个即将停用的热点
    lost = X_val[hotspots].ne(100).sum(axis=1)
    return lost[lost > 0].mean()

rng = np.random.default_rng(42)
random_means = []

for _ in range(20):
    hotspots = rng.choice(available_hotspots, size=20, replace=False).tolist()
    random_means.append(mean_lost_signals(hotspots))

top20 = detection_counts.nlargest(20).index.tolist()

print(f"随机停用 20 个：每条受影响扫描平均丢 {np.mean(random_means):.2f} 个信号")
print(f"常见热点停用 20 个：每条受影响扫描平均丢 {mean_lost_signals(top20):.2f} 个信号")

随机停用 20 个：每条受影响扫描平均丢 1.50 个信号
常见热点停用 20 个：每条受影响扫描平均丢 6.65 个信号


In [12]:
rng_aug = np.random.default_rng(42)

# 随机选一半训练扫描，只遮掉原本能看到的信号
selected_rows = rng_aug.choice(len(X_train), size=len(X_train) // 2, replace=False)
mask_rows = np.zeros(len(X_train), dtype=bool)
mask_rows[selected_rows] = True
visible = X_train.to_numpy() != 100
hide = mask_rows[:, None] & visible & (rng_aug.random(X_train.shape) < 0.20)

X_train_aug = X_train_model.mask(hide, -110)

aug_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
aug_model.fit(X_train_aug, y_train)

# 两个模型看到的是同一批模拟故障记录
X_val_outage = X_val_model.copy()
X_val_outage.loc[:, top20] = -110

comparison = []
for condition, inputs in [
    ("原始扫描", X_val_model),
    ("停用 20 个常见热点", X_val_outage),
]:
    baseline = 100 * accuracy_score(y_val, model.predict(inputs))
    augmented = 100 * accuracy_score(y_val, aug_model.predict(inputs))
    comparison.append({
        "验证条件": condition,
        "普通训练（%）": baseline,
        "遮挡训练（%）": augmented,
        "差值（百分点）": augmented - baseline,
    })

pd.DataFrame(comparison).round(2)

,验证条件,普通训练（%）,遮挡训练（%）,差值（百分点）
0,原始扫描,96.97,96.46,-0.50
1,停用 20 个常见热点,90.12,91.27,1.15


In [13]:
def compare_models(hotspots):
    inputs = X_val_model.copy()
    inputs.loc[:, hotspots] = -110

    return np.array([
        100 * accuracy_score(y_val, model.predict(inputs)),
        100 * accuracy_score(y_val, aug_model.predict(inputs)),
    ])

rows = [("原始扫描", 0, *compare_models([]))]

for k in [5, 10, 20]:
    # 沿用基线实验里的随机种子、抽样范围和轮数
    rng = np.random.default_rng(42)
    random_scores = []
    for _ in range(20):
        hotspots = rng.choice(available_hotspots, size=k, replace=False).tolist()
        random_scores.append(compare_models(hotspots))

    rows.append(("随机停用（20 组平均）", k, *np.mean(random_scores, axis=0)))

    common_hotspots = detection_counts.nlargest(k).index.tolist()
    rows.append(("停用常见热点", k, *compare_models(common_hotspots)))

comparison_table = pd.DataFrame(
    rows, columns=["验证条件", "停用热点数", "普通训练（%）", "遮挡训练（%）"]
)
comparison_table["差值（百分点）"] = (
    comparison_table["遮挡训练（%）"] - comparison_table["普通训练（%）"]
)
comparison_table.round(2)

,验证条件,停用热点数,普通训练（%）,遮挡训练（%）,差值（百分点）
0,原始扫描,0,96.97,96.46,-0.50
1,随机停用（20 组平均）,5,96.73,96.32,-0.41
2,停用常见热点,5,96.51,95.91,-0.60
3,随机停用（20 组平均）,10,96.31,96.11,-0.20
4,停用常见热点,10,94.48,93.30,-1.18
5,随机停用（20 组平均）,20,95.39,95.60,0.21
6,停用常见热点,20,90.12,91.27,1.15


In [14]:
# 重现上一轮的随机顺序，只把遮挡概率从 20% 改为 10%
rng_aug10 = np.random.default_rng(42)
selected_rows10 = rng_aug10.choice(
    len(X_train), size=len(X_train) // 2, replace=False
)
mask_rows10 = np.zeros(len(X_train), dtype=bool)
mask_rows10[selected_rows10] = True
hide10 = (
    mask_rows10[:, None]
    & (X_train.to_numpy() != 100)
    & (rng_aug10.random(X_train.shape) < 0.10)
)

aug_model10 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
aug_model10.fit(X_train_model.mask(hide10, -110), y_train)

def score_aug10(hotspots):
    inputs = X_val_model.copy()
    inputs.loc[:, hotspots] = -110
    return 100 * accuracy_score(y_val, aug_model10.predict(inputs))

scores10 = [score_aug10([])]
for k in [5, 10, 20]:
    rng10 = np.random.default_rng(42)
    random_scores10 = []
    for _ in range(20):
        hotspots = rng10.choice(available_hotspots, size=k, replace=False).tolist()
        random_scores10.append(score_aug10(hotspots))

    scores10.append(np.mean(random_scores10))
    common_hotspots = detection_counts.nlargest(k).index.tolist()
    scores10.append(score_aug10(common_hotspots))

rate_comparison = comparison_table.iloc[:, :4].copy()
rate_comparison = rate_comparison.rename(columns={"遮挡训练（%）": "20% 遮挡（%）"})
rate_comparison.insert(3, "10% 遮挡（%）", scores10)
rate_comparison.round(2)

,验证条件,停用热点数,普通训练（%）,10% 遮挡（%）,20% 遮挡（%）
0,原始扫描,0,96.97,96.76,96.46
1,随机停用（20 组平均）,5,96.73,96.58,96.32
2,停用常见热点,5,96.51,96.21,95.91
3,随机停用（20 组平均）,10,96.31,96.40,96.11
4,停用常见热点,10,94.48,94.01,93.30
5,随机停用（20 组平均）,20,95.39,95.80,95.60
6,停用常见热点,20,90.12,91.17,91.27


In [15]:
# 固定验证输入，只改变训练时随机遮掉哪些信号
checks = [
    ("原始扫描", X_val_model),
    ("停用 20 个常见热点", X_val_outage),
]

seed_results = []
for seed in [42, 7, 123]:
    rng_seed = np.random.default_rng(seed)
    chosen = rng_seed.choice(len(X_train), size=len(X_train) // 2, replace=False)
    chosen_rows = np.zeros(len(X_train), dtype=bool)
    chosen_rows[chosen] = True

    visible_in_chosen = chosen_rows[:, None] & (X_train.to_numpy() != 100)
    random_draws = rng_seed.random(X_train.shape)

    for rate, existing_model in [(0.10, aug_model10), (0.20, aug_model)]:
        if seed == 42:
            trained_model = existing_model
        else:
            hidden = visible_in_chosen & (random_draws < rate)
            trained_model = make_pipeline(
                StandardScaler(), LogisticRegression(max_iter=1000)
            )
            trained_model.fit(X_train_model.mask(hidden, -110), y_train)

        result = {"随机种子": seed, "遮挡概率": f"{rate:.0%}"}
        for name, inputs in checks:
            result[name + "（%）"] = 100 * accuracy_score(
                y_val, trained_model.predict(inputs)
            )
        seed_results.append(result)

pd.DataFrame(seed_results).round(2)

,随机种子,遮挡概率,原始扫描（%）,停用 20 个常见热点（%）
0,42,10%,96.76,91.17
1,42,20%,96.46,91.27
2,7,10%,96.76,91.20
3,7,20%,97.04,90.90
4,123,10%,97.09,88.96
5,123,20%,97.09,89.64


In [16]:
cv_results = []

for fold, (train_ids, val_ids) in enumerate(
    splitter.split(X, y, groups=location_group), start=1
):
    X_fit_raw = X.iloc[train_ids]
    X_fit = X_fit_raw.replace(100, -110)
    y_fit = y.iloc[train_ids]
    X_clean = X.iloc[val_ids].replace(100, -110)
    y_check = y.iloc[val_ids]

    visible = X_fit_raw.ne(100)
    fold_top20 = visible.sum().nlargest(20).index.tolist()
    X_failed = X_clean.copy()
    X_failed.loc[:, fold_top20] = -110

    def fit_and_score(training_inputs):
        fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        fitted.fit(training_inputs, y_fit)
        return np.array([
            accuracy_score(y_check, fitted.predict(X_clean)),
            accuracy_score(y_check, fitted.predict(X_failed)),
        ]) * 100

    scores = {
        "普通训练": [fit_and_score(X_fit)],
        "10% 遮挡": [],
        "20% 遮挡": [],
    }

    for seed in [42, 7, 123]:
        rng_fold = np.random.default_rng(seed)
        selected = rng_fold.choice(len(X_fit), size=len(X_fit) // 2, replace=False)
        eligible = np.zeros(len(X_fit), dtype=bool)
        eligible[selected] = True
        draws = rng_fold.random(X_fit.shape)

        for rate in [0.10, 0.20]:
            hidden = eligible[:, None] & visible.to_numpy() & (draws < rate)
            scores[f"{rate:.0%} 遮挡"].append(fit_and_score(X_fit.mask(hidden, -110)))

    for method, runs in scores.items():
        clean_acc, failed_acc = np.mean(runs, axis=0)
        cv_results.append((fold, method, clean_acc, failed_acc))

cv_table = pd.DataFrame(
    cv_results, columns=["折", "训练方式", "原始扫描（%）", "停用 20 个常见热点（%）"]
)
cv_table.round(2)

,折,训练方式,原始扫描（%）,停用 20 个常见热点（%）
0,1,普通训练,96.97,90.12
1,1,10% 遮挡,96.87,90.44
2,1,20% 遮挡,96.86,90.60
3,2,普通训练,96.40,86.25
4,2,10% 遮挡,95.11,85.21
5,2,20% 遮挡,94.53,85.14
6,3,普通训练,98.26,88.66
7,3,10% 遮挡,98.27,87.73
8,3,20% 遮挡,97.90,87.74
9,4,普通训练,97.19,88.34


## 最终测试

内部五折验证后，选定普通逻辑回归作为主模型。用完整的 `trainingData.csv` 重新拟合标准化和分类器，用官方提供的 `validationData.csv` 进行独立测试。沿用已设定的 0、5、10、20 个热点停用条件。热点选择只使用训练集。每档随机停用使用种子 42，抽取 20 组热点，汇报平均结果。

In [17]:
# 用所有训练记录拟合最终基线模型
final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_model.fit(X.replace(100, -110), y)

# 官方预留文件只用于最后打分
final_data = pd.read_csv("validationData.csv")
X_final_raw = final_data[wifi_columns]
X_final = X_final_raw.replace(100, -110)
y_final = (
    final_data["BUILDINGID"].astype(str) + "_" + final_data["FLOOR"].astype(str)
)

final_detection_counts = X.ne(100).sum()
final_available = final_detection_counts[final_detection_counts > 0].index
print(f"训练记录：{len(X):,} 条，独立测试记录：{len(X_final):,} 条。")

训练记录：19,937 条，独立测试记录：1,111 条。


In [18]:
def score_final_outage(hotspots):
    inputs = X_final.copy()
    inputs.loc[:, hotspots] = -110
    accuracy = 100 * accuracy_score(y_final, final_model.predict(inputs))
    affected = X_final_raw[hotspots].ne(100).any(axis=1).sum()
    return accuracy, affected

clean_accuracy, clean_affected = score_final_outage([])
final_rows = [("原始扫描", 0, clean_accuracy, clean_affected)]

for k in [5, 10, 20]:
    rng_test = np.random.default_rng(42)
    random_scores = [
        score_final_outage(rng_test.choice(final_available, size=k, replace=False).tolist())
        for _ in range(20)
    ]
    random_accuracy, random_affected = np.mean(random_scores, axis=0)
    final_rows.append(("随机停用（20 组平均）", k, random_accuracy, random_affected))

    common = final_detection_counts.nlargest(k).index.tolist()
    common_accuracy, common_affected = score_final_outage(common)
    final_rows.append(("停用常见热点", k, common_accuracy, common_affected))

final_table = pd.DataFrame(
    final_rows,
    columns=["测试条件", "停用热点数", "准确率（%）", "受影响的扫描数"],
)
final_table.insert(3, "比原始下降（百分点）", clean_accuracy - final_table["准确率（%）"])
final_table.round(2)

,测试条件,停用热点数,准确率（%）,比原始下降（百分点）,受影响的扫描数
0,原始扫描,0,88.21,0.00,0.00
1,随机停用（20 组平均）,5,88.03,0.18,135.65
2,停用常见热点,5,87.76,0.45,212.00
3,随机停用（20 组平均）,10,87.70,0.50,327.30
4,停用常见热点,10,86.05,2.16,366.00
5,随机停用（20 组平均）,20,87.25,0.95,550.40
6,停用常见热点,20,82.99,5.22,391.00
